# DS2 Enrichment A: ISLP Lab 4.7 on CADets, and Loss Functions

**Pairs with:** ISLP Ch. 4, sections 4.7.2 (Lab: Logistic Regression) and 4.3.2 (Estimating the Regression Coefficients)

**Time:** ~45 min, optional, self-paced

**What you will do**
- Walk through the logistic-regression part of ISLP Lab 4.7, using CADets admissions instead of stock-market data
- Fit the same model two ways (`smf.logit` and `sm.GLM`) and see that they agree
- Turn predicted probabilities into labels, build a confusion table, and see why accuracy at a 0.5 cutoff misleads when only 9% of patients die
- Split the data into training and test sets, as ISLP does, and refit with fewer predictors
- See what "maximum likelihood" means by plotting the log loss and finding its minimum, and compare log loss with squared error

In [ ]:
# Setup: load the CADets dataset (release 1)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf

DATA_FILE = "cadets_admissions_release1_core.csv"
DATA_URL = "https://raw.githubusercontent.com/Rotem-Gershon/cadets-notebooks/main/data/" + DATA_FILE

# Use a local copy if there is one; otherwise read the course copy from GitHub.
df = pd.read_csv(DATA_FILE if os.path.exists(DATA_FILE) else DATA_URL)
print(df.shape)

## Part 1: ISLP Lab 4.7, logistic regression on CADets

### 1.1 The outcome and its prevalence

ISLP predicts whether the stock market goes up or down. Our outcome is `in_hospital_death` (1 = died during the admission). Look at the share of deaths: it is small, and that will matter when we judge accuracy later.

In [ ]:
# How many patients died in hospital?
print(df["in_hospital_death"].value_counts())
print(f"Proportion died: {df['in_hospital_death'].mean():.3f}")

### 1.2 Fit the multivariable model with `smf.logit`

This is the same model as the core DS2 lesson: in-hospital death predicted from age, sex, background diseases, and admission vitals and labs. The formula interface handles `sex` (F/M) for us. Look at the table: one row per predictor, with its coefficient (`coef`, on the log-odds scale) and its p-value (`P>|z|`).

In [ ]:
formula = ("in_hospital_death ~ age + sex + heart_failure + diabetes + ckd + copd"
           " + active_cancer + heart_rate + systolic_bp + resp_rate + spo2"
           " + creatinine + hemoglobin + wbc")

model_logit = smf.logit(formula, data=df).fit(disp=0)  # disp=0 hides fitting messages
print(model_logit.summary())

### 1.3 The same model with `sm.GLM`, as in ISLP

ISLP fits logistic regression as a *generalized linear model* (GLM) with a binomial family. ISLP builds the predictor matrix with its own `ModelSpec` tool; we build it by hand with pandas. The only manual step is turning `sex` into a 0/1 column (`male`). Then we add a column of 1s for the intercept.

In [ ]:
import statsmodels.api as sm

predictors = ["age", "heart_failure", "diabetes", "ckd", "copd", "active_cancer",
              "heart_rate", "systolic_bp", "resp_rate", "spo2",
              "creatinine", "hemoglobin", "wbc"]
X = df[predictors].copy()
X["male"] = (df["sex"] == "M").astype(int)   # sex as a 0/1 column
X = sm.add_constant(X)                       # intercept column of 1s
y = df["in_hospital_death"]

model_glm = sm.GLM(y, X, family=sm.families.Binomial()).fit()

Now put the two sets of coefficients side by side. They should match to several decimal places: logistic regression is a GLM, so the two functions solve the same problem.

In [ ]:
compare = pd.DataFrame({
    "smf.logit": model_logit.params.rename({"sex[T.M]": "male"}),
    "sm.GLM": model_glm.params.rename({"const": "Intercept"}),
}).dropna()
print(compare.round(4))

### 1.4 Reading the results: coefficients, odds ratios, p-values

ISLP looks at `params` (the coefficients) and `pvalues`. Exponentiating a coefficient gives the adjusted odds ratio. Check that heart failure gives about 1.47, the same adjusted OR as in the core lesson.

In [ ]:
results = pd.DataFrame({
    "coef": model_glm.params,
    "odds_ratio": np.exp(model_glm.params),
    "p_value": model_glm.pvalues,
})
print(results.drop(index="const").round(3))

### 1.5 Predicted probabilities

`predict()` returns each patient's predicted probability of death (a number between 0 and 1). ISLP prints the first ten; so do we.

In [ ]:
probs = model_glm.predict(X)   # predicted probability of death for every patient
print(probs[:10].round(3))
print(f"Mean predicted probability: {probs.mean():.3f}")

### 1.6 From probabilities to labels, and the confusion table

To get a yes/no prediction, ISLP uses a cutoff of 0.5: above it, predict "died". `pd.crosstab` then counts how predictions line up with what actually happened. Rows are predictions, columns are the truth.

In [ ]:
labels = np.where(probs > 0.5, "Died", "Survived")
truth = np.where(y == 1, "Died", "Survived")

print(pd.crosstab(labels, truth, rownames=["Predicted"], colnames=["Actual"]))
print(f"Accuracy: {np.mean(labels == truth):.3f}")

### 1.7 Why 0.5 is a poor cutoff when only 9% die

The accuracy looks impressive, but compare it with a "model" that predicts *every* patient survives. Because 91% of patients survive, that useless rule is already about 91% accurate. Look at the confusion table again: most patients who died were predicted to survive.

With a rare outcome, a 0.5 cutoff almost never flags anyone. In practice the cutoff is a clinical and product decision (how many alerts can the ward handle, how many deaths can we afford to miss?). DS3 covers this properly with sensitivity, specificity and ROC curves.

In [ ]:
always_survive = np.mean(y == 0)
print(f"Accuracy if we predict 'survived' for everyone: {always_survive:.3f}")

is_death = y == 1
caught = np.mean(probs[is_death] > 0.5)
print(f"Share of deaths flagged at cutoff 0.5: {caught:.2f}")

**Try it.** Repeat the confusion table with a cutoff of 0.2 instead of 0.5 (change `0.5` to `0.2`). How many more deaths are flagged, and how many extra survivors are flagged as well? Is accuracy higher or lower?

In [ ]:
# Your code here

### 1.8 Training and test sets

So far we judged the model on the same patients it learned from, which flatters it. ISLP splits by date (train on 2001 to 2004, test on 2005). Release 1 of CADets has no admission date, so we use a random 70/30 split instead. Fixing `random_state` means everyone gets the same split.

In [ ]:
train = df.sample(frac=0.7, random_state=1)   # 70% of patients, chosen at random
test = df.drop(train.index)                   # the remaining 30%

print(f"Train: {len(train)} patients, death rate {train['in_hospital_death'].mean():.3f}")
print(f"Test:  {len(test)} patients, death rate {test['in_hospital_death'].mean():.3f}")

Fit the full model on the training patients only, then predict for the test patients it has never seen. We write a small helper so we can reuse it for the next model.

In [ ]:
def test_confusion(formula, cutoff=0.5):
    fit = smf.logit(formula, data=train).fit(disp=0)       # learn on training set
    p_test = fit.predict(test)                              # predict on test set
    pred = np.where(p_test > cutoff, "Died", "Survived")
    actual = np.where(test["in_hospital_death"] == 1, "Died", "Survived")
    print(pd.crosstab(pred, actual, rownames=["Predicted"], colnames=["Actual"]))
    print(f"Test accuracy: {np.mean(pred == actual):.3f}")

test_confusion(formula)

### 1.9 Refit with a smaller set of predictors

ISLP notices that most predictors have large p-values and refits with only the two strongest ones. We do the same in spirit: keep five predictors with clear clinical meaning and small p-values in section 1.4 (age, heart failure, oxygen saturation, respiratory rate, creatinine). A smaller model is easier to explain, cheaper to deploy (fewer data feeds), and sometimes predicts just as well.

In [ ]:
small_formula = "in_hospital_death ~ age + heart_failure + spo2 + resp_rate + creatinine"
test_confusion(small_formula)

Accuracy alone barely moves, because at a 0.5 cutoff both models mostly predict "survived". To really compare them we need tools from DS3 (AUC, calibration). For now, note that dropping nine predictors cost very little.

**Try it.** Run `test_confusion(small_formula, cutoff=0.2)` and `test_confusion(formula, cutoff=0.2)`. At this cutoff, does the smaller model catch about as many deaths as the full one?

In [ ]:
# Your code here

## Part 2: Loss functions and maximum likelihood (ISLP 4.3.2)

### 2.1 What "fitting" means

How does the computer choose the coefficients? It tries values and keeps the ones that make the observed outcomes most likely. This is *maximum likelihood*. Equivalently, it minimizes the **log loss** (also called cross-entropy): the average of minus log(predicted probability of what actually happened). A confident wrong prediction gets a large penalty.

We use the simple model from the core lesson, death ~ heart failure. Its fitted slope is about 1.024 (OR 2.78).

In [ ]:
simple = smf.logit("in_hospital_death ~ heart_failure", data=df).fit(disp=0)
b0_hat, b1_hat = simple.params["Intercept"], simple.params["heart_failure"]
print(f"Fitted intercept b0 = {b0_hat:.3f}, slope b1 = {b1_hat:.3f}, OR = {np.exp(b1_hat):.2f}")

### 2.2 A log-loss function

This function takes a candidate intercept and slope, computes every patient's predicted probability, and returns the average log loss. Lower is better.

In [ ]:
hf = df["heart_failure"].to_numpy()
died = df["in_hospital_death"].to_numpy()

def log_loss(b0, b1):
    p = 1 / (1 + np.exp(-(b0 + b1 * hf)))                     # predicted probability
    return -np.mean(died * np.log(p) + (1 - died) * np.log(1 - p))

print(f"Log loss at the fitted values: {log_loss(b0_hat, b1_hat):.4f}")
print(f"Log loss if b1 = 0 (heart failure ignored): {log_loss(b0_hat, 0):.4f}")

### 2.3 Search over slopes by hand

Keep the intercept at its fitted value and try 4,001 slopes between -1 and 3, in steps of 0.001. Plot the log loss for each. The lowest point of the curve should sit exactly on the fitted slope (dashed line).

In [ ]:
b1_grid = np.round(np.arange(-1, 3.0005, 0.001), 3)   # -1.000, -0.999, ..., 3.000
losses = [log_loss(b0_hat, b1) for b1 in b1_grid]
best_b1 = b1_grid[np.argmin(losses)]

plt.plot(b1_grid, losses)
plt.axvline(b1_hat, color="gray", linestyle="--", label=f"fitted b1 = {b1_hat:.3f}")
plt.xlabel("Slope b1 for heart failure (log-odds)"); plt.ylabel("Average log loss")
plt.legend(); plt.show()
print(f"Grid minimum at b1 = {best_b1:.3f} (OR {np.exp(best_b1):.2f})")

### 2.4 (Optional) Both coefficients at once

Now vary the intercept and the slope together. The contour plot is a map of the log loss: like a valley seen from above. The fitted pair (the cross) sits at the bottom of the valley.

In [ ]:
b0_grid = np.linspace(-3.2, -2.0, 80)
b1_grid2 = np.linspace(0, 2, 80)
Z = np.array([[log_loss(b0, b1) for b0 in b0_grid] for b1 in b1_grid2])

plt.contour(b0_grid, b1_grid2, Z, levels=20)
plt.plot(b0_hat, b1_hat, "kx", markersize=10)
plt.xlabel("Intercept b0"); plt.ylabel("Slope b1"); plt.title("Log loss landscape")
plt.show()

### 2.5 Squared error vs. log loss for single predictions

In DS1 (linear regression) we minimized *squared error*. Why not do the same here? Compare the two penalties for a few predictions. The table shows patients who **died** (y = 1) and patients who **survived** (y = 0), with different predicted probabilities of death.

In [ ]:
examples = pd.DataFrame({
    "actual_y":  [1,   1,   1,   1,    0,   0],
    "predicted": [0.8, 0.5, 0.1, 0.01, 0.1, 0.8],
})
p, yv = examples["predicted"], examples["actual_y"]
examples["squared_error"] = (yv - p) ** 2
examples["log_loss"] = -(yv * np.log(p) + (1 - yv) * np.log(1 - p))
print(examples.round(3))

Look at the row where the patient died but the model said 1%. Squared error can never exceed 1, so it barely distinguishes "very wrong" from "badly wrong". Log loss grows without limit as a confident prediction becomes more wrong: a model that confidently tells the ward a dying patient is safe is punished hard.

### 2.6 The two loss curves for a patient who died

For a patient with y = 1, plot both penalties against the predicted probability.

In [ ]:
p_grid = np.linspace(0.01, 0.99, 200)
plt.plot(p_grid, (1 - p_grid) ** 2, label="Squared error")
plt.plot(p_grid, -np.log(p_grid), label="Log loss")
plt.xlabel("Predicted probability of death (patient died, y = 1)")
plt.ylabel("Penalty"); plt.legend(); plt.show()

Both curves fall to zero as the prediction approaches the truth, but log loss shoots up as the prediction approaches 0. Log loss is also what maximum likelihood minimizes, which gives the familiar standard errors and p-values. That is why logistic regression is fit with log loss, and why you will see "log loss" or "cross-entropy" in almost every AI model card for a binary outcome.

**Try it.** Add a row to the examples table for a survivor (`actual_y` = 0) predicted at 0.99. How large is each penalty? Which loss would push a model harder to avoid this mistake?

In [ ]:
# Your code here

## Key points

- `smf.logit` and `sm.GLM(..., family=sm.families.Binomial())` fit the same logistic regression; ISLP uses the GLM form.
- Predicted probabilities become labels only after choosing a cutoff. With 9% mortality, a 0.5 cutoff flags almost no one, and accuracy looks high even for a useless model.
- Judge a model on patients it did not learn from (a test set). Without dates, a random split with a fixed seed is a reasonable start.
- "Maximum likelihood" means choosing the coefficients that minimize log loss; the minimum of the curve sits exactly at the fitted coefficient (1.024 for heart failure).
- Log loss punishes confident wrong predictions far more than squared error, which suits probability models.

**Further reading:** ISLP Ch. 4, section 4.3.2 (maximum likelihood) and section 4.7.2 (Lab: Logistic Regression).